In [31]:
# Functions to scrape data from wikipedia
import requests
from requests.adapters import HTTPAdapter
from urllib3 import Retry
import random

class ScrapingException(BaseException):
    """Errors gotten when trying to scrape from wikipedia"""

# Use different user agents to mimic different browsers
USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
]

def get_session():
    """
    Function to return a session that I can use to make requests to wikipedia's pages
    """
    session = requests.Session()
    
    # Add retry logic
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter=adapter)
    return session
    
SESSION = get_session()

def get_wikipedia_page(url):
    """
    A function to try getting a wikipedia page
    """
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    # Make a request to Wikipedia
    try:
        print("Getting response")
        wikipedia_response = SESSION.get(url, headers=headers, timeout=15)
        if wikipedia_response.status_code != 200:
            raise ScrapingException(f"Wikipedia refused to send page with status code {wikipedia_response.status_code}")
        else:
            return wikipedia_response
    except Exception as e:
        print(f"Could not get page because of {e}")
        raise Exception("Unknown error")
    
jean_paul_response = get_wikipedia_page("https://en.wikipedia.org/wiki/Jean-Paul_Abalo")

Getting response


In [67]:
from bs4 import BeautifulSoup
import re

class PlayerDetails:
    def __init__(self, from_year: int | None, to_year: int | None, appearances: int | None, goals: int | None, team: str | None):
        self.from_year = from_year
        self.to_year = to_year
        self.appearances = appearances
        self.goals = goals
        self.team = team
        
    def __repr__(self):
        return f"Found data row with from: {self.from_year} to: {self.to_year} team: {self.team} appearances: {self.appearances} and goals: {self.goals}"

def extract_infobox(player_wikipedia: str, player_id: str, problematic_players: list[str]) -> list[PlayerDetails] | None:
    """
    Extract player details from a player's wikipedia page
    
    Args:
        player_wikipedia: URL of a player's wikipedia page
        player_id: id of player being processed
        problematic_players: list of players whose wikipedia page couldn't be parsed
        
    Returns:
        players: A list of player objects if response of player was parsable
    """
    try:    
        player_response = get_wikipedia_page(player_wikipedia)
        assert player_response.status_code == 200, "Can't parse an unsucessful response"
        
        # Parse HTML of the player's Wiki page
        soup = BeautifulSoup(player_response.text, 'html.parser')

        # Look for the players info box
        tables = soup.select_one("table.infobox")

        # Store unparsable player for later processing
        if not tables:
            print(f"Player does not have info box in wiki: {player_wikipedia}")
            problematic_players.append(player_id)
            return None

        player_details = []
        table_rows = tables.find_all("tr")
        current = 0
        section_name = ""
        content_num = 0

        # Scanning table rows
        while True:
            if current >= len(table_rows):
                print("Consumed all rows in table")
                break
            
            current_row = table_rows[current]
            
            # Checking if row is the heading of a section
            if len(current_row.find_all("td")) == 0:
                section_name = current_row.select_one("th").text
                content_num = 0
                
                # Check if it's the section we want
                if "Senior career" in section_name:
                    # While next item is content
                    while current + 1 < len(table_rows) and len(table_rows[current + 1].find_all("td")) > 0:
                        next_content = table_rows[current + 1]
                        # Check if it is a table row
                        if len(next_content.find_all("b")) > 0:
                            print(f"Found title row {next_content.find("b")}")
                        else:
                            # Work with data that looks like 1 th 3 td
                            data_timeline = next_content.find("th")
                            
                            from_year = None
                            to_year = None
                            team = None
                            appearances = None
                            goals = None
                            
                            # Extract from and to years
                            if data_timeline:
                                raw_timeline = data_timeline.get_text(strip=True)
                                if len(raw_timeline) > 4:
                                    from_year = int(re.sub(r'\D', '', raw_timeline[0:4]))
                                    if len(raw_timeline) > 5:
                                        to_year = int(re.sub(r'\D', '', raw_timeline[4:]))
                                else:
                                    from_year = int(re.sub(r'\D', '', raw_timeline))
                            
                            rest_data = next_content.find_all("td")
                            if len(rest_data) < 3:
                                print(f"Unkown rest data {rest_data}")
                                problematic_players.append(player_id)
                            # Get team, appearance, goals
                            else:
                                raw_appearance = rest_data[1].get_text(strip=True)
                                raw_goals = rest_data[2].get_text(strip=True)
                                team = rest_data[0].get_text(strip=True)
                                if len(raw_appearance) > 0 and raw_appearance != "?":
                                    appearances = int(re.sub(r'\D', '', raw_appearance))
                                if len(raw_goals) > 0 and raw_goals != "(?)":
                                    goals = int(re.sub(r'\D', '', raw_goals))
                        
                            print(f"Found data row with from: {from_year} to: {to_year} team: {team} appearances: {appearances} and goals: {goals}")
                            player_details.append(PlayerDetails(
                                from_year=from_year,
                                to_year=to_year,
                                appearances=appearances,
                                goals=goals,
                                team = team
                            ))
                            content_num += 1
                        
                        current += 1
                        
                    # Increment
                    current += 1
                    print(f"Section {section_name} had {content_num} items")
                else:
                    current += 1
            else:
                current += 1
                
        return player_details
    except Exception as e:
        print(f"Unexpected error {e}")
        problematic_players.append(player_id)
    except ScrapingException as e:
        print(f"Error getting Wikipedia page {e}")
        problematic_players.append(player_id)
        
problematic_players = []
player_info = extract_infobox("https://en.wikipedia.org/wiki/Jean-Paul_Abalo", 'P001', problematic_players)
print(player_info)

Getting response
Found title row <b>Team</b>
Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None
Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appearances: 29 and goals: 1
Found data row with from: 1995 to: 2005 team: Amiens SC appearances: 273 and goals: 7
Found data row with from: 2005 to: None team: USL Dunkerque appearances: 4 and goals: 0
Found data row with from: 2006 to: None team: APOEL appearances: 3 and goals: 0
Found data row with from: 2006 to: None team: Ethnikos Piraeus appearances: 9 and goals: 0
Found data row with from: 2007 to: 2008 team: Al-Merrikh appearances: None and goals: None
Found data row with from: 2008 to: 2009 team: FC Déols 36 appearances: None and goals: None
Section Senior career* had 8 items
Consumed all rows in table
[Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None, Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appear

In [53]:
class Player:
    def __init__(self, id: str, wikipedia_url: str, problematic_players: list[str]):
        self.id = id
        self.wikipedia_url = wikipedia_url
        self.wikipedia_details = self._extract_wikipedia_details(problematic_players=problematic_players)
        
    def _extract_wikipedia_details(self, problematic_players: list[str]) -> list[PlayerDetails] | None:
        """
        Returns player details that have been extracted from their wikipedia page
        
        Args:
            problematic_players : A list used to store all player ids that had an issue extracting details from
            
        Returns:
            player_details: A list of player details from wikipedia
        """
        return extract_infobox(self.wikipedia_url, self.id, problematic_players)
    
    def to_csv(self) -> list[str]:
        """ 
        Returns strings that will represent the player in a CSV file
        
        Returns:
            entries (str): Entries in CSV file
        """
        entries = []
        if self.wikipedia_details:
            for detail in self.wikipedia_details:
                entries.append(f"{self.id},{detail.appearances},{detail.team},{detail.goals},{detail.from_year},{detail.to_year}\n")
            
        return entries
    
    def __repr__(self):
        return f"Player: {self.wikipedia_details}"

In [49]:
raw_players = [
    {'id': 'P001', 'url': 'https://en.wikipedia.org/wiki/Alan_A%27Court'},
    {'id': 'P002', 'url': 'https://en.wikipedia.org/wiki/Stefan_Abadzhiev'},
    {'id': 'P003', 'url': 'https://en.wikipedia.org/wiki/Patrice_Abanda'}
]

players = []

for raw in raw_players:
    players.append(
        Player(raw['id'], wikipedia_url=raw['url'], problematic_players=problematic_players)
    )
    
print(players)

Getting response
Found title row <b>Team</b>
Found data row with from: 1952 to: 1964 team: Liverpool appearances: 354 and goals: 61
Found data row with from: 1964 to: 1966 team: Tranmere Rovers appearances: 50 and goals: 11
Found data row with from: 1966 to: 1967 team: Norwich City appearances: 0 and goals: 0
Found title row <b>404</b>
Section Senior career* had 3 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1953 to: 1968 team: Levski Sofia appearances: 254 and goals: 37
Found data row with from: 1968 to: 1970 team: Wiesbaden appearances: None and goals: None
Section Senior career* had 2 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1995 to: 1998 team: Tonnerre Yaoundé appearances: 0 and goals: 0
Found data row with from: 1998 to: 1999 team: PAOK appearances: 0 and goals: 0
Found data row with from: 1999 to: 2000 team: Apollon Kalamarias appearances: 0 and goals: 0
Found

In [50]:
from datetime import datetime
def export_players_csv(players: list[Player]):
    """
    Function that will store each of the player details in a newly generated CSV file
    """
    file_name = f"players_{datetime.now()}.csv"
    
    with open(file_name, 'a') as file:
        # Insert csv file header
        file.write("id,appearances,team,goals,from,to\n")
        
        # Insert entries in CSV
        for player in players:
            entries = player.to_csv()
            for entry in entries:
                file.write(entry)
    
export_players_csv(players)

In [51]:
import pandas as pd

# Import list of players
df = pd.read_csv('players.csv')
df.shape

(7907, 12)

In [54]:
import math, time

batch_size = 100
total_batches = math.ceil(df.shape[0] / batch_size)
print(f"Total batches: {total_batches}")
extracted_players = []
problematic_players = []

file_name = f"players_{datetime.now()}.csv"
    
with open(file_name, 'a') as file:
    # Insert csv file header
    file.write("id,appearances,team,goals,from,to\n")
    
    # Get players
    for batch_num in range(total_batches):
        # Getting indexes of batch
        print(f"Processing batch #{batch_num}")
        start = batch_num * batch_size
        if batch_num == total_batches - 1:
            end = df.shape[0]
        else:
            end = start + batch_size
           
        # Storing details of players in batch 
        df_segment = df.iloc[start:end]
        for row in df_segment.itertuples():            
            player = Player(
                id=row.player_id,
                wikipedia_url=row.player_wikipedia_link,
                problematic_players=problematic_players
            )
            
            extracted_players.append(player)
            # Insert entries in CSV
            for entry in player.to_csv():
                file.write(entry)
            
        time.sleep(10)
    

Total batches: 80
Processing batch #0
Getting response
Found title row <b>Team</b>
Found data row with from: 1952 to: 1964 team: Liverpool appearances: 354 and goals: 61
Found data row with from: 1964 to: 1966 team: Tranmere Rovers appearances: 50 and goals: 11
Found data row with from: 1966 to: 1967 team: Norwich City appearances: 0 and goals: 0
Found title row <b>404</b>
Section Senior career* had 3 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1953 to: 1968 team: Levski Sofia appearances: 254 and goals: 37
Found data row with from: 1968 to: 1970 team: Wiesbaden appearances: None and goals: None
Section Senior career* had 2 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None
Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appearances: 29 and goals: 1
Found data row with from: 1995 to: 20

In [56]:
print(f"Number of players: {len(extracted_players)}")
print(f"Number of problematic players -> {problematic_players} with {len(problematic_players)} entries")


Number of players: 7907
Number of problematic players -> ['P-06687', 'P-08180', 'P-04364', 'P-08688', 'P-00061', 'P-03821', 'P-07032', 'P-00912', 'P-05355', 'P-05203', 'P-08988', 'P-03870', 'P-02968', 'P-05758', 'P-08418', 'P-08254', 'P-09997', 'P-07137', 'P-08649', 'P-03329', 'P-04700', 'P-04538', 'P-00834', 'P-01832', 'P-07871', 'P-01613', 'P-09250', 'P-05826', 'P-05015', 'P-08579', 'P-05231', 'P-03366', 'P-09480', 'P-00025', 'P-04959', 'P-06195', 'P-05438', 'P-07471', 'P-00776', 'P-07744', 'P-00932', 'P-05463', 'P-08801', 'P-08079', 'P-00209', 'P-04315', 'P-09121', 'P-00944', 'P-01056', 'P-01349', 'P-06236', 'P-08058', 'P-08466', 'P-06738', 'P-00433', 'P-08669', 'P-05010', 'P-00917', 'P-02421', 'P-09305', 'P-08772', 'P-05397', 'P-09598', 'P-07583', 'P-06309', 'P-05620', 'P-00460', 'P-01128', 'P-09301', 'P-08849', 'P-07896', 'P-04857', 'P-02072', 'P-01434', 'P-09615', 'P-07737', 'P-01884', 'P-06707', 'P-08269', 'P-03121', 'P-05128', 'P-02792', 'P-04748', 'P-05121', 'P-01450', 'P-0178

In [59]:
problematic_players_df = df[df['player_id'].isin(problematic_players)]
problematic_players_df

,key_id,player_id,family_name,given_name,birth_date,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
10,11,P-06687,Abdel Rahman,Adel,1967-12-11,0,0,0,1,1,1990,https://en.wikipedia.org/wiki/Adel_Abdel_Rahman
20,21,P-08180,Abdullahi,Shehu,1993-03-12,0,1,0,0,1,2018,https://en.wikipedia.org/wiki/Shehu_Abdullahi
24,25,P-04364,Abedzadeh,Amir,1993-04-26,1,0,0,0,1,2018,https://en.wikipedia.org/wiki/Amir_Abedzadeh
29,30,P-08688,Abeledo,Ramón,1937-04-29,0,0,1,0,1,1962,https://en.wikipedia.org/wiki/Ram%C3%B3n_Abeledo
34,35,P-00061,Aboubakar,Vincent,1992-01-22,0,0,0,1,2,"2010, 2014",https://en.wikipedia.org/wiki/Vincent_Aboubakar
...,...,...,...,...,...,...,...,...,...,...,...,...
7890,7891,P-01914,Zomers,Hendrikus,not available,0,0,0,1,1,1938,https://en.wikipedia.org/wiki/Hendrikus_V._%22...
7893,7894,P-07883,Zsak,Manfred,1964-12-22,0,0,1,0,1,1990,https://en.wikipedia.org/wiki/Manfred_Zsak
7895,7896,P-04170,Zuber,Steven,1991-08-17,0,0,1,0,1,2018,https://en.wikipedia.org/wiki/Steven_Zuber
7897,7898,P-07832,Zubia,Oscar,1946-02-08,0,0,0,1,1,1970,https://en.wikipedia.org/wiki/Oscar_Zubia


In [61]:
diagnostic_sample = problematic_players_df.sample(n=10)
for row in diagnostic_sample.itertuples():
    print(row.player_wikipedia_link)
    player = Player(
                id=row.player_id,
                wikipedia_url=row.player_wikipedia_link,
                problematic_players=problematic_players
            )

https://en.wikipedia.org/wiki/Gustav_Wetterstr%C3%B6m
Getting response
Found title row <b>Team</b>
Unexpected error invalid literal for int() with base 10: ''
https://en.wikipedia.org/wiki/Abdelkader_Horr
Getting response
Found title row <b>Team</b>
Unexpected error invalid literal for int() with base 10: ''
https://en.wikipedia.org/wiki/Li_Chi-an
Getting response
Player does not have info box in wiki: https://en.wikipedia.org/wiki/Li_Chi-an
https://en.wikipedia.org/wiki/Jordan_Ayew
Getting response
Found title row <b>Team</b>
Found data row with from: 2009 to: 2014 team: Marseille appearances: 111 and goals: 14
Found data row with from: 2014 to: None team: →Sochaux(loan) appearances: 17 and goals: 5
Found data row with from: 2014 to: 2015 team: Lorient appearances: 31 and goals: 12
Found data row with from: 2015 to: 2017 team: Aston Villa appearances: 51 and goals: 9
Found data row with from: 2017 to: 2019 team: Swansea City appearances: 50 and goals: 8
Found data row with from: 2018 

In [68]:
# Fix for player with no to date
player_wiki = "https://en.wikipedia.org/wiki/Jordan_Ayew"
extract_infobox(player_wiki, "test", [])

Getting response
Found title row <b>Team</b>
Found data row with from: 2009 to: 2014 team: Marseille appearances: 111 and goals: 14
Found data row with from: 2014 to: None team: →Sochaux(loan) appearances: 17 and goals: 5
Found data row with from: 2014 to: 2015 team: Lorient appearances: 31 and goals: 12
Found data row with from: 2015 to: 2017 team: Aston Villa appearances: 51 and goals: 9
Found data row with from: 2017 to: 2019 team: Swansea City appearances: 50 and goals: 8
Found data row with from: 2018 to: 2019 team: →Crystal Palace(loan) appearances: 20 and goals: 1
Found data row with from: 2019 to: 2024 team: Crystal Palace appearances: 175 and goals: 21
Found data row with from: 2024 to: None team: Leicester City appearances: 72 and goals: 11
Section Senior career* had 8 items
Consumed all rows in table


[Found data row with from: 2009 to: 2014 team: Marseille appearances: 111 and goals: 14,
 Found data row with from: 2014 to: None team: →Sochaux(loan) appearances: 17 and goals: 5,
 Found data row with from: 2014 to: 2015 team: Lorient appearances: 31 and goals: 12,
 Found data row with from: 2015 to: 2017 team: Aston Villa appearances: 51 and goals: 9,
 Found data row with from: 2017 to: 2019 team: Swansea City appearances: 50 and goals: 8,
 Found data row with from: 2018 to: 2019 team: →Crystal Palace(loan) appearances: 20 and goals: 1,
 Found data row with from: 2019 to: 2024 team: Crystal Palace appearances: 175 and goals: 21,
 Found data row with from: 2024 to: None team: Leicester City appearances: 72 and goals: 11]

In [69]:
# Fix for data with question marks
player_wiki = "https://en.wikipedia.org/wiki/Rosen_Kirilov"
extract_infobox(player_wiki, "test", [])

Getting response
Found title row <b>Team</b>
Found data row with from: 1990 to: 1991 team: Bdin Vidin appearances: None and goals: None
Found data row with from: 1991 to: 1996 team: CSKA Sofia appearances: 61 and goals: 2
Found data row with from: 1994 to: None team: →Spartak Pleven(loan) appearances: None and goals: None
Found data row with from: 1996 to: None team: →Litex Lovech(loan) appearances: 12 and goals: 1
Found data row with from: 1996 to: 1998 team: Litex Lovech appearances: 39 and goals: 1
Found data row with from: 1999 to: 2001 team: Adanaspor appearances: 63 and goals: 0
Found data row with from: 2001 to: 2007 team: Litex Lovech appearances: 122 and goals: 10
Found data row with from: 2007 to: 2008 team: APOP Kinyras appearances: 22 and goals: 0
Found data row with from: 2008 to: None team: Vaslui appearances: 1 and goals: 0
Found title row <b>327</b>
Section Senior career* had 9 items
Consumed all rows in table


[Found data row with from: 1990 to: 1991 team: Bdin Vidin appearances: None and goals: None,
 Found data row with from: 1991 to: 1996 team: CSKA Sofia appearances: 61 and goals: 2,
 Found data row with from: 1994 to: None team: →Spartak Pleven(loan) appearances: None and goals: None,
 Found data row with from: 1996 to: None team: →Litex Lovech(loan) appearances: 12 and goals: 1,
 Found data row with from: 1996 to: 1998 team: Litex Lovech appearances: 39 and goals: 1,
 Found data row with from: 1999 to: 2001 team: Adanaspor appearances: 63 and goals: 0,
 Found data row with from: 2001 to: 2007 team: Litex Lovech appearances: 122 and goals: 10,
 Found data row with from: 2007 to: 2008 team: APOP Kinyras appearances: 22 and goals: 0,
 Found data row with from: 2008 to: None team: Vaslui appearances: 1 and goals: 0]

In [70]:
# Run after major fixes
problematic_players_2 = []
batch_size = 100
total_batches = math.ceil(problematic_players_df.shape[0] / batch_size)
print(f"Total batches: {total_batches}")
extracted_players_2 = []

file_name = f"problematic_players_{datetime.now()}.csv"
    
with open(file_name, 'a') as file:
    # Insert csv file header
    file.write("id,appearances,team,goals,from,to\n")
    
    # Get players
    for batch_num in range(total_batches):
        # Getting indexes of batch
        print(f"Processing batch #{batch_num}")
        start = batch_num * batch_size
        if batch_num == total_batches - 1:
            end = df.shape[0]
        else:
            end = start + batch_size
           
        # Storing details of players in batch 
        df_segment = problematic_players_df.iloc[start:end]
        for row in df_segment.itertuples():            
            player = Player(
                id=row.player_id,
                wikipedia_url=row.player_wikipedia_link,
                problematic_players=problematic_players_2
            )
            
            extracted_players_2.append(player)
            # Insert entries in CSV
            for entry in player.to_csv():
                file.write(entry)
            
        time.sleep(10)

Total batches: 16
Processing batch #0
Getting response
Found title row <b>Team</b>
Found data row with from: 1987 to: 1988 team: Olympic Club appearances: None and goals: None
Found data row with from: 1988 to: 1995 team: Al Ahly appearances: None and goals: None
Found data row with from: 1995 to: 1995 team: Zamalek SC appearances: None and goals: None
Found data row with from: 1995 to: 1997 team: Al Wehda FC appearances: None and goals: None
Section Senior career* had 4 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 2012 to: 2014 team: Kano Pillars appearances: None and goals: None
Found data row with from: 2014 to: 2015 team: Qadsia appearances: 7 and goals: 1
Found data row with from: 2015 to: 2016 team: União da Madeira appearances: 28 and goals: 1
Found data row with from: 2016 to: 2018 team: Anorthosis appearances: 48 and goals: 3
Found data row with from: 2018 to: 2020 team: Bursaspor appearances: 60 and goals: 5
Found dat

In [72]:
print(f"Number of players: {len(extracted_players_2)}")
print(f"Number of problematic players -> {problematic_players_2} with {len(problematic_players_2)} entries")

Number of players: 1552
Number of problematic players -> ['P-03821', 'P-00912', 'P-05203', 'P-03870', 'P-02968', 'P-08254', 'P-07137', 'P-08649', 'P-03329', 'P-04700', 'P-04538', 'P-00834', 'P-05015', 'P-00025', 'P-08801', 'P-08079', 'P-00209', 'P-09121', 'P-01349', 'P-06738', 'P-00433', 'P-08669', 'P-05010', 'P-00917', 'P-09305', 'P-05397', 'P-01128', 'P-09301', 'P-08849', 'P-04857', 'P-01434', 'P-09615', 'P-07737', 'P-03121', 'P-02792', 'P-04893', 'P-09068', 'P-02382', 'P-07625', 'P-08711', 'P-00507', 'P-01960', 'P-07829', 'P-03581', 'P-03972', 'P-09800', 'P-08383', 'P-09160', 'P-07588', 'P-01316', 'P-02403', 'P-06051', 'P-05521', 'P-00831', 'P-02420', 'P-08010', 'P-01683', 'P-05779', 'P-00965', 'P-01714', 'P-07531', 'P-02362', 'P-09378', 'P-01244', 'P-09226', 'P-09381', 'P-04527', 'P-02093', 'P-03000', 'P-03191', 'P-09883', 'P-03256', 'P-00684', 'P-09254', 'P-07406', 'P-09815', 'P-03395', 'P-06174', 'P-06546', 'P-07298', 'P-07563', 'P-08414', 'P-06512', 'P-01932', 'P-09038', 'P-0509

In [74]:
problematic_players_df_2 = df[df['player_id'].isin(problematic_players_2)]
diagnostic_sample = problematic_players_df_2.sample(n=10)
for row in diagnostic_sample.itertuples():
    print(row.player_wikipedia_link)
    player = Player(
                id=row.player_id,
                wikipedia_url=row.player_wikipedia_link,
                problematic_players=problematic_players
            )

https://en.wikipedia.org/wiki/Mih%C3%A1ly_B%C3%ADr%C3%B3
Getting response
Found title row <b>Team</b>
Found data row with from: 1937 to: 1940 team: Ferencvárosi TC appearances: 26 and goals: 7
Found title row <b>Runner-up</b>
Unkown rest data [<td colspan="2" style="padding:0">
</td>]
Found data row with from: None to: None team: None appearances: None and goals: None
Section Senior career* had 2 items
Consumed all rows in table
https://en.wikipedia.org/wiki/Sergei_Gorlukovich
Getting response
Found title row <b>Team</b>
Unexpected error invalid literal for int() with base 10: ''
https://en.wikipedia.org/wiki/Kim_Yong-dae
Getting response
Player does not have info box in wiki: https://en.wikipedia.org/wiki/Kim_Yong-dae
https://en.wikipedia.org/wiki/Karel_%C4%8Cern%C3%BD_(footballer)
Getting response
Found title row <b>Team</b>
Found data row with from: 1932 to: 1933 team: SK Plzeň appearances: None and goals: None
Found data row with from: 1933 to: 1937 team: SK Židenice appearances: N